# TinyLlama Nepali Alpaca QLoRA Fine-Tuning

This notebook fine-tunes `TinyLlama/TinyLlama-1.1B-Chat-v1.0` on `saillab/alpaca-nepali-cleaned` using 4-bit QLoRA. It includes:

- a train/test split and conversational prompt construction;
- inference with the untouched base model;
- precision-safe LoRA initialization for T4 and newer NVIDIA GPUs;
- training/evaluation loss and next-token accuracy tracking;
- a live plot saved throughout training;
- final inference, adapter saving, reloading, and ZIP export.

> **Important:** Token accuracy is teacher-forced next-token accuracy. It is useful for monitoring training, but it is not the same as factual, semantic, or instruction-following accuracy.

## 1. Install dependencies

Run this once in a fresh Kaggle session. After installation, restart the kernel once and then run the notebook from Section 2 in order. The Matplotlib upper bound avoids a conflict with Kaggle's preinstalled `ydata-profiling`.

In [ ]:
%pip install -q -U transformers datasets accelerate bitsandbytes peft trl "matplotlib<3.11"

## 2. Imports, reproducibility, and hardware check

BF16 is used only on Ampere-or-newer GPUs. A Tesla T4 uses FP16. LoRA parameters will later be kept in FP32 to avoid AMP gradient-unscaling errors.

In [ ]:
import gc
import inspect
import json
import math
import os
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import transformers
import trl
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainerCallback,
    set_seed,
)
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTConfig, SFTTrainer

set_seed(42)

if not torch.cuda.is_available():
    raise RuntimeError('This QLoRA notebook requires an NVIDIA CUDA GPU.')

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MAJOR, GPU_MINOR = torch.cuda.get_device_capability(0)
USE_BF16 = GPU_MAJOR >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('TRL:', trl.__version__)
print('GPU:', GPU_NAME)
print('Compute capability:', GPU_MAJOR, GPU_MINOR)
print('QLoRA compute dtype:', COMPUTE_DTYPE)

## 3. Configuration

The effective batch size is `2 × 8 = 16` examples on one GPU. Start with one epoch as a functional experiment, then increase only after inspecting validation loss and generated answers.

In [ ]:
MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
DATASET_NAME = 'saillab/alpaca-nepali-cleaned'
OUTPUT_DIR = '/kaggle/working/tinyllama-nepali-alpaca-qlora'
MAX_LENGTH = 512
NUM_EPOCHS = 1
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
TEST_SIZE = 0.02
SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 4. Load and inspect the dataset

Each source example contains an instruction, optional input, and output. The split is deterministic because a fixed seed is used.

In [ ]:
raw_dataset = load_dataset(DATASET_NAME, split='train')
print(raw_dataset)
print('Columns:', raw_dataset.column_names)
print('First example:', raw_dataset[0])

In [ ]:
dataset_split = raw_dataset.train_test_split(
    test_size=TEST_SIZE,
    seed=SEED,
)

def clean_text(value):
    # Some rows contain missing values represented as NaN or the string 'nan'.
    if value is None:
        return ''
    try:
        if bool(np.isnan(value)):
            return ''
    except (TypeError, ValueError):
        pass
    text = str(value).strip()
    return '' if text.lower() in {'nan', 'none', 'null'} else text

def convert_to_conversation(example):
    instruction = clean_text(example.get('instruction'))
    inp = clean_text(example.get('input'))
    output = clean_text(example.get('output'))

    if inp:
        user_content = f'{instruction}\n\nथप जानकारी:\n{inp}'
    else:
        user_content = instruction

    return {
        'prompt': [{'role': 'user', 'content': user_content}],
        'completion': [{'role': 'assistant', 'content': output}],
    }

conversation_dataset = dataset_split.map(
    convert_to_conversation,
    remove_columns=raw_dataset.column_names,
)
conversation_dataset = conversation_dataset.filter(
    lambda row: bool(row['prompt'][0]['content'])
    and bool(row['completion'][0]['content'])
)

print(conversation_dataset)
print(json.dumps(
    conversation_dataset['train'][0],
    ensure_ascii=False,
    indent=2,
))

## 5. Load tokenizer and 4-bit base model

NF4 quantization stores the frozen base model compactly. Double quantization saves additional memory. On a T4, all quantized computation is explicitly FP16.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tiny_tok = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

if tiny_tok.pad_token is None:
    tiny_tok.pad_token = tiny_tok.eos_token
tiny_tok.padding_side = 'right'

tiny_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={'': 0},
    dtype=COMPUTE_DTYPE,
)

tiny_model.config.pad_token_id = tiny_tok.pad_token_id
tiny_model.config.use_cache = True

print('pad_token:', repr(tiny_tok.pad_token))
print('pad_token_id:', tiny_tok.pad_token_id)

## 6. Base-model inference before fine-tuning

This response is your baseline. Save it so that the post-training comparison uses exactly the same prompt and decoding configuration.

In [ ]:
SAMPLE_INSTRUCTION = 'नेपालको राजधानी कहाँ हो? यसको छोटो परिचय नेपाली भाषामा दिनुहोस्।'

@torch.inference_mode()
def generate_response(model, tokenizer, instruction, max_new_tokens=128):
    messages = [{'role': 'user', 'content': instruction}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    encoded = tokenizer(
        prompt,
        return_tensors='pt',
    ).to(model.device)

    torch.manual_seed(SEED)
    generated = model.generate(
        **encoded,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

    answer_ids = generated[0, encoded['input_ids'].shape[1]:]
    return tokenizer.decode(
        answer_ids,
        skip_special_tokens=True,
    ).strip()

tiny_model.eval()
base_answer = generate_response(
    tiny_model,
    tiny_tok,
    SAMPLE_INSTRUCTION,
)
print('BASE MODEL RESPONSE:\n')
print(base_answer)

## 7. Prepare QLoRA adapters safely

The base model stays frozen and quantized. LoRA adapters are inserted into attention and MLP projections. Only trainable adapters are upcast to FP32. Frozen non-quantized layers retain the precision established by prepare_model_for_kbit_training; do not downcast them with a model-wide casting utility.

In [ ]:
tiny_model.config.use_cache = False
tiny_model = prepare_model_for_kbit_training(
    tiny_model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)

tiny_model = get_peft_model(
    tiny_model,
    lora_config,
    autocast_adapter_dtype=True,
)
for parameter in tiny_model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()
tiny_model.print_trainable_parameters()

trainable_dtypes = Counter(
    str(parameter.dtype)
    for parameter in tiny_model.parameters()
    if parameter.requires_grad
)
bf16_trainable = [
    name for name, parameter in tiny_model.named_parameters()
    if parameter.requires_grad and parameter.dtype == torch.bfloat16
]

print('Trainable dtypes:', trainable_dtypes)
print('BF16 trainable tensors:', len(bf16_trainable))
assert set(trainable_dtypes) == {'torch.float32'}, trainable_dtypes
assert not bf16_trainable, bf16_trainable[:10]

## 8. Use TRL's native next-token accuracy

TRL 1.12 computes `mean_token_accuracy` directly inside `SFTTrainer` after shifting causal-language-model labels and masking ignored tokens. Therefore, this notebook intentionally does not attach a custom `compute_metrics` or `preprocess_logits_for_metrics` function. Custom preprocessing conflicts with TRL's auxiliary scalar outputs in this version.

In [ ]:
print('Accuracy source: TRL SFTTrainer native mean_token_accuracy')
print('Custom compute_metrics: disabled')

## 9. Live loss and accuracy plot

Training loss and evaluation loss use the left axis. Accuracy uses a separate 0–100% right axis. The callback overwrites the PNG and JSON history after each logging event, so partial results survive an interrupted run.

In [ ]:
class TrainingCurveCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.figure_path = os.path.join(output_dir, 'training_curves.png')
        self.history_path = os.path.join(output_dir, 'training_metrics.json')
        self.records = []
        os.makedirs(output_dir, exist_ok=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not state.is_world_process_zero or not logs:
            return
        record = {
            'step': int(state.global_step),
            'epoch': float(state.epoch) if state.epoch is not None else None,
        }
        record.update({
            key: float(value)
            for key, value in logs.items()
            if isinstance(value, (int, float))
        })
        self.records.append(record)
        self.save()

    def on_train_end(self, args, state, control, **kwargs):
        if state.is_world_process_zero:
            self.save()

    def save(self):
        with open(self.history_path, 'w', encoding='utf-8') as file:
            json.dump(self.records, file, indent=2)
        self.plot()

    def plot(self):
        train_points = [x for x in self.records if 'loss' in x]
        eval_points = [x for x in self.records if 'eval_loss' in x]
        train_accuracy_points = [
            x for x in self.records if 'mean_token_accuracy' in x
        ]
        eval_accuracy_points = [
            x for x in self.records if 'eval_mean_token_accuracy' in x
        ]
        if not (
            train_points or eval_points
            or train_accuracy_points or eval_accuracy_points
        ):
            return

        plt.close('all')
        figure, loss_axis = plt.subplots(figsize=(10, 6), dpi=150)
        accuracy_axis = loss_axis.twinx()

        if train_points:
            loss_axis.plot(
                [x['step'] for x in train_points],
                [x['loss'] for x in train_points],
                color='#2563EB', linewidth=2, label='Training loss',
            )
        if eval_points:
            loss_axis.plot(
                [x['step'] for x in eval_points],
                [x['eval_loss'] for x in eval_points],
                color='#F97316', linewidth=2, marker='o',
                label='Evaluation loss',
            )
        if train_accuracy_points:
            accuracy_axis.plot(
                [x['step'] for x in train_accuracy_points],
                [100 * x['mean_token_accuracy'] for x in train_accuracy_points],
                color='#16A34A', linewidth=2,
                label='Training token accuracy',
            )
        if eval_accuracy_points:
            accuracy_axis.plot(
                [x['step'] for x in eval_accuracy_points],
                [100 * x['eval_mean_token_accuracy'] for x in eval_accuracy_points],
                color='#9333EA', linewidth=2, marker='s',
                label='Evaluation token accuracy',
            )

        loss_axis.set_xlabel('Optimizer step')
        loss_axis.set_ylabel('Cross-entropy loss', color='#1D4ED8')
        accuracy_axis.set_ylabel('Token accuracy (%)', color='#15803D')
        accuracy_axis.set_ylim(0, 100)
        loss_axis.tick_params(axis='y', labelcolor='#1D4ED8')
        accuracy_axis.tick_params(axis='y', labelcolor='#15803D')
        loss_axis.grid(True, alpha=0.25, linestyle='--')
        loss_axis.set_title('TinyLlama Nepali QLoRA Training')

        lines_1, labels_1 = loss_axis.get_legend_handles_labels()
        lines_2, labels_2 = accuracy_axis.get_legend_handles_labels()
        loss_axis.legend(lines_1 + lines_2, labels_1 + labels_2, loc='best')
        figure.tight_layout()
        figure.savefig(self.figure_path, bbox_inches='tight')
        plt.close(figure)

curve_callback = TrainingCurveCallback(OUTPUT_DIR)

## 10. Build a version-tolerant SFT configuration

Evaluation frequency is derived from the expected optimizer-step count, producing approximately ten validation measurements. Unsupported optional arguments are filtered by inspecting the installed `SFTConfig` signature.

In [ ]:
training_device_count = max(1, torch.cuda.device_count())
effective_batch_size = (
    TRAIN_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
    * training_device_count
)
steps_per_epoch = math.ceil(
    len(conversation_dataset['train']) / effective_batch_size
)
total_steps = max(1, math.ceil(steps_per_epoch * NUM_EPOCHS))
warmup_steps = max(1, int(0.03 * total_steps))
eval_steps = max(1, min(200, total_steps // 10))

config_values = dict(
    output_dir=OUTPUT_DIR,
    max_length=MAX_LENGTH,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    fp16=not USE_BF16,
    bf16=USE_BF16,
    max_grad_norm=1.0,
    logging_steps=10,
    logging_nan_inf_filter=False,
    eval_strategy='steps',
    eval_steps=eval_steps,
    eval_accumulation_steps=4,
    save_strategy='steps',
    save_steps=eval_steps,
    save_total_limit=2,
    report_to='none',
    seed=SEED,
    remove_unused_columns=False,
    packing=False,
    completion_only_loss=True,
)

supported = inspect.signature(SFTConfig.__init__).parameters
filtered_values = {
    key: value for key, value in config_values.items() if key in supported
}
ignored_values = sorted(set(config_values) - set(filtered_values))
tiny_sft_config = SFTConfig(**filtered_values)

print('Visible training GPUs:', training_device_count)
print('Effective batch size:', effective_batch_size)
print('Expected optimizer steps:', total_steps)
print('Warmup steps:', warmup_steps)
print('Evaluate/save every:', eval_steps, 'steps')
print('Ignored unsupported options:', ignored_values)

## 11. Create the trainer

LoRA is already attached, so `peft_config` must not be passed again. After trainer construction, the adapters are cast to FP32 one final time because some TRL/Transformers combinations recast them while preparing the dataset. This placement is essential on a Tesla T4.

In [ ]:
tiny_trainer = SFTTrainer(
    model=tiny_model,
    args=tiny_sft_config,
    train_dataset=conversation_dataset['train'],
    eval_dataset=conversation_dataset['test'],
    processing_class=tiny_tok,
    callbacks=[curve_callback],
)

# Ensures eval_loss is returned for PEFT models in affected versions.
tiny_trainer.can_return_loss = True
tiny_trainer.compute_metrics = None
tiny_trainer.preprocess_logits_for_metrics = None

# Run this AFTER SFTTrainer construction. It is intentionally not earlier.
for parameter in tiny_trainer.model.parameters():
    if parameter.requires_grad and parameter.dtype != torch.float32:
        parameter.data = parameter.data.to(torch.float32)
        parameter.grad = None

trainer_trainable_dtypes = Counter(
    str(parameter.dtype)
    for parameter in tiny_trainer.model.parameters()
    if parameter.requires_grad
)
remaining_bf16 = [
    name for name, parameter in tiny_trainer.model.named_parameters()
    if parameter.requires_grad and parameter.dtype == torch.bfloat16
]
print('Prepared training rows:', len(tiny_trainer.train_dataset))
print('Prepared evaluation rows:', len(tiny_trainer.eval_dataset))
print('Trainer trainable dtypes:', trainer_trainable_dtypes)
print('Remaining BF16 trainable tensors:', len(remaining_bf16))
assert len(tiny_trainer.train_dataset) > 0, 'No trainable examples remain.'
assert set(trainer_trainable_dtypes) == {'torch.float32'}, trainer_trainable_dtypes
assert not remaining_bf16, remaining_bf16[:10]
tiny_trainer.model.print_trainable_parameters()

## 12. Train, evaluate, plot, and save

The callback saves `training_curves.png` and `training_metrics.json` after every log. Checkpoints are also saved periodically.

In [ ]:
print('FP16:', tiny_trainer.args.fp16)
print('BF16:', tiny_trainer.args.bf16)
print('Accelerate precision:', tiny_trainer.accelerator.mixed_precision)
print('Scaler:', tiny_trainer.accelerator.scaler)

preflight_dtypes = {
    parameter.dtype for parameter in tiny_trainer.model.parameters()
    if parameter.requires_grad
}
assert preflight_dtypes == {torch.float32}, preflight_dtypes

from transformers.trainer_utils import get_last_checkpoint

# Explicit opt-in: do not automatically reload a potentially non-finite run.
last_checkpoint = None  # Set a known-good checkpoint path to resume.

# Evaluate before spending time on training. Abort on invalid loss.
initial_metrics = tiny_trainer.evaluate()
print('Pre-training evaluation:', initial_metrics)
if not math.isfinite(float(initial_metrics.get('eval_loss', float('nan')))):
    raise RuntimeError('Non-finite initial evaluation loss. Diagnose before training.')

train_result = tiny_trainer.train(
    resume_from_checkpoint=last_checkpoint or False
)
final_metrics = tiny_trainer.evaluate()

tiny_trainer.log_metrics('train', train_result.metrics)
tiny_trainer.save_metrics('train', train_result.metrics)
tiny_trainer.log_metrics('final_eval', final_metrics)
tiny_trainer.save_metrics('final_eval', final_metrics)
tiny_trainer.save_state()
tiny_trainer.save_model(OUTPUT_DIR)
tiny_tok.save_pretrained(OUTPUT_DIR)
curve_callback.save()

print('Training complete.')
print('Final evaluation:', final_metrics)
print('Saved to:', OUTPUT_DIR)

In [ ]:
from IPython.display import Image, display

display(Image(filename=curve_callback.figure_path))

## 13. Compare the fine-tuned model with the saved baseline

Generation settings and the prompt remain unchanged, making the comparison more meaningful.

In [ ]:
tiny_trainer.model.eval()
tiny_trainer.model.config.use_cache = True

fine_tuned_answer = generate_response(
    tiny_trainer.model,
    tiny_tok,
    SAMPLE_INSTRUCTION,
)

print('=' * 80)
print('PROMPT:\n', SAMPLE_INSTRUCTION)
print('=' * 80)
print('BASE MODEL RESPONSE:\n', base_answer)
print('=' * 80)
print('FINE-TUNED MODEL RESPONSE:\n', fine_tuned_answer)

## 14. Reload the saved adapter in a fresh session

Use this section later when you want inference without retraining. It reloads the same quantized base model and then places the saved LoRA adapter on top.

In [ ]:
# Run this in a fresh runtime after executing the import/configuration cells.
reload_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
reload_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={'': 0},
    dtype=COMPUTE_DTYPE,
)
reloaded_model = PeftModel.from_pretrained(
    reload_base,
    OUTPUT_DIR,
)
reloaded_model.eval()

print(generate_response(
    reloaded_model,
    reload_tokenizer,
    SAMPLE_INSTRUCTION,
))

## 15. Create a downloadable ZIP archive

In [ ]:
import shutil

archive_path = shutil.make_archive(
    OUTPUT_DIR,
    'zip',
    root_dir=OUTPUT_DIR,
)
print('Archive created:', archive_path)

print('Download the ZIP from Kaggle Outputs:', archive_path)

## Reading the curves

- **Training loss decreases:** the adapters are learning the training examples.
- **Evaluation loss decreases:** generalization to held-out examples is improving.
- **Training loss falls while evaluation loss rises:** likely overfitting; stop earlier or reduce epochs.
- **Token accuracy rises:** next-token predictions are improving on the held-out completion tokens.
- **Good loss but poor generated answers:** inspect prompt formatting, response truncation, dataset quality, and qualitative evaluation prompts.

For research reporting, complement token accuracy with held-out loss/perplexity, Nepali-specific evaluation tasks, and human or model-assisted response-quality judgments.